# LangChain 멀티모달리티(Multimodality) - v1.0 

---

## 1. 멀티모달리티 개요

- **멀티모달리티(Multimodality)** 는 텍스트, 이미지, 오디오, 비디오 등 다양한 형태의 데이터를 처리하는 기술

- **LangChain의 주요 멀티모달 컴포넌트:**

    - **채팅 모델(Chat Models)**: 다양한 입출력 형식 처리
    - **임베딩 모델(Embedding Models)**: 다양한 데이터 타입의 벡터 표현
    - **벡터 저장소(Vector Stores)**: 멀티모달 데이터의 임베딩 검색


---

## 2. 표준 콘텐츠 블록 (Standard Content Blocks)

### 2.1 메시지 콘텐츠 형식

- LangChain 채팅 모델은 `content` 속성에서 다음 세 가지 형식을 지원합니다:

    1. **문자열**: 단순 텍스트 메시지
    2. **프로바이더 네이티브 포맷**: 프로바이더별 형식
    3. **표준 콘텐츠 블록**: LangChain 표준 형식 (권장 ✅)

        ```python
        from langchain.messages import HumanMessage

        # 1. 문자열 콘텐츠
        message1 = HumanMessage("안녕하세요")

        # 2. 프로바이더 네이티브 포맷 (OpenAI)
        message2 = HumanMessage(content=[
            {"type": "text", "text": "안녕하세요"},
            {"type": "image_url", "image_url": {"url": "https://example.com/image.jpg"}}
        ])

        # 3. 표준 콘텐츠 블록 (권장)
        message3 = HumanMessage(content_blocks=[
            {"type": "text", "text": "안녕하세요"},
            {"type": "image", "url": "https://example.com/image.jpg"}
        ])
        ```

### 2.2 `content_blocks` 속성

- 모든 메시지 객체는 `content_blocks` 속성을 통해 **표준화된 타입 안전 표현**에 접근 가능

    ```python
    from langchain.chat_models import init_chat_model

    model = init_chat_model("gpt-4o")
    response = model.invoke("AI에 대해 설명해주세요")

    # content_blocks를 통한 표준화된 접근
    for block in response.content_blocks:
        if block["type"] == "text":
            print(block["text"])
        elif block["type"] == "reasoning":
            print(f"추론 과정: {block['reasoning']}")
    ```

---

## 3. 멀티모달 콘텐츠 블록 타입

### 3.1 지원하는 콘텐츠 블록 타입

- **Core 블록**

    | 타입 | 설명 | 주요 필드 |
    |------|------|-----------|
    | `text` | 표준 텍스트 출력 | `text`, `annotations` |
    | `reasoning` | 모델 추론 단계 | `reasoning` |

<br>

- **Multimodal 블록**

    | 타입 | 설명 | 주요 필드 |
    |------|------|-----------|
    | `image` | 이미지 데이터 | `url`, `base64`, `id`, `mime_type` |
    | `audio` | 오디오 데이터 | `url`, `base64`, `id`, `mime_type` |
    | `video` | 비디오 데이터 | `url`, `base64`, `id`, `mime_type` |
    | `file` | 일반 파일 (PDF 등) | `url`, `base64`, `id`, `mime_type` |
    | `text-plain` | 문서 텍스트 (.txt, .md) | `text`, `mime_type` |

<br>

- **Tool Calling 블록**

    | 타입 | 설명 |
    |------|------|
    | `tool_call` | 함수 호출 |
    | `tool_call_chunk` | 스트리밍 도구 호출 단편 |
    | `invalid_tool_call` | 잘못된 도구 호출 |

### 3.2 데이터 전달 방식

- **URL 방식**

    ```python
    image_block = {
        "type": "image",
        "url": "https://example.com/image.jpg",
        "mime_type": "image/jpeg"
    }
    ```

    - **장점:**
        - 간단하고 직관적
        - 대용량 파일에 유리

    - **단점:**
        - 외부 접근 가능한 URL 필요
        - 네트워크 의존성

- **Base64 인코딩 방식 (권장)**

    ```python
    import base64

    def encode_image_to_base64(image_path: str) -> str:
        """이미지를 base64로 인코딩"""
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode("utf-8")

    image_block = {
        "type": "image",
        "base64": encode_image_to_base64("photo.jpg"),
        "mime_type": "image/jpeg"
    }
    ```

    - **장점:**
        - 가장 안정적
        - URL 불필요
        - 모든 프로바이더 지원

    - **단점:**
        - 토큰 사용량 증가

- **File ID 방식 (프로바이더 관리)**

    ```python
    image_block = {
        "type": "image",
        "file_id": "file-abc123",  # 프로바이더가 관리하는 파일 ID
    }
    ```

    - **장점:**
        - 토큰 효율적
        - 재사용 가능

    - **단점:**
        - 프로바이더별 사전 업로드 필요
        - 제한적 지원

---

## 4. 이미지 처리 실습

### 4.1 환경 설정

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
import base64
import requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

import warnings
warnings.filterwarnings('ignore')

`(3) langfuase handler 설정`

In [ ]:
from langfuse.langchain import CallbackHandler

# 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

### 4.2 모델 초기화 (v1.0 방식)

In [ ]:
# init_chat_model 사용 (권장)
model = init_chat_model(
    "gpt-4.1-mini",  # 멀티모달 지원 모델
    temperature=0
)

In [ ]:
# 또는 특정 프로바이더 클래스 사용
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

### 4.3 URL 방식 이미지 처리

In [ ]:
def process_image_from_url(image_url: str, prompt: str):
    """URL에서 직접 이미지 처리 (v1.0 표준 콘텐츠 블록)"""
    
    message = HumanMessage(content_blocks=[
        {"type": "text", "text": prompt},
        {"type": "image", "url": image_url, "mime_type": "image/jpeg"}
    ])
    
    return model.invoke([message])

# 실행 예시
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
result = process_image_from_url(image_url, "이미지를 상세히 설명해주세요")
print(result.content)

In [ ]:
# URL이 입력된 경우를 지원하는 이미지 출력 함수
import requests
from io import BytesIO
import matplotlib.pyplot as plt
from PIL import Image

def show_image(image_path):
    if isinstance(image_path, str) and image_path.startswith("http"):
        # URL에서 이미지 다운로드
        headers = {
            'User-Agent': 'Mozilla/5.0'
        }
        response = requests.get(image_path, headers=headers)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content))
    else:
        image = Image.open(image_path)
    plt.imshow(image)
    plt.axis('off')
    plt.show()

show_image(image_url)

### 4.4 Base64 방식 이미지 처리

In [ ]:
def encode_image_to_base64(image_path: str) -> str:
    """로컬 이미지를 base64로 인코딩"""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def encode_image_url_to_base64(image_url: str) -> str:
    """URL 이미지를 base64로 인코딩"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    response = requests.get(image_url, headers=headers)
    response.raise_for_status()
    return base64.b64encode(response.content).decode('utf-8')

def process_image_with_base64(image_source: str, prompt: str):
    """Base64로 이미지 처리 (v1.0 표준 콘텐츠 블록)"""
    
    # 이미지를 base64로 변환
    if image_source.startswith("http"):
        image_data = encode_image_url_to_base64(image_source)
    else:
        image_data = encode_image_to_base64(image_source)
    
    message = HumanMessage(content_blocks=[
        {"type": "text", "text": prompt},
        {
            "type": "image",
            "base64": image_data,
            "mime_type": "image/jpeg"
        }
    ])
    
    return model.invoke([message])

# 실행 예시
result = process_image_with_base64("portrait.jpg", "이 초상화의 특징을 분석해주세요")
print(result.content)

### 4.5 여러 이미지 동시 처리

In [ ]:
def process_multiple_images(image_sources: list, prompt: str):
    """여러 이미지를 동시에 처리 (v1.0 표준 콘텐츠 블록)"""
    
    # 콘텐츠 블록 리스트 시작 (프롬프트)
    content_blocks = [{"type": "text", "text": prompt}]
    
    # 각 이미지를 콘텐츠 블록에 추가
    for image_source in image_sources:
        if image_source.startswith("http"):
            # URL 처리
            image_data = encode_image_url_to_base64(image_source)
            content_blocks.append({
                "type": "image",
                "base64": image_data,
                "mime_type": "image/jpeg"
            })
        else:
            # Base64 직접 처리
            image_data = encode_image_to_base64(image_source)
            content_blocks.append({
                "type": "image",
                "base64": image_data,
                "mime_type": "image/jpeg"
            })
    
    message = HumanMessage(content_blocks=content_blocks)
    return model.invoke([message])

# 실행 예시
images = [
    "portrait.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg/600px-Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg"
]

result = process_multiple_images(
    images,
    "이 두 작품을 비교 분석해주세요. 화풍, 색채, 구도의 차이점을 중심으로 설명해주세요."
)
print(result.content)

---

## 5. 다양한 멀티모달 타입 처리

### 5.1 PDF 문서 처리

In [ ]:
def process_pdf_document(pdf_source: str, prompt: str):
    """PDF 문서 처리 (v1.0 표준 콘텐츠 블록)"""
    
    if pdf_source.startswith("http"):
        # URL 방식
        content_blocks = [
            {"type": "text", "text": prompt},
            {"type": "file", "url": pdf_source, "mime_type": "application/pdf"}
        ]
    else:
        # Base64 방식
        with open(pdf_source, "rb") as pdf_file:
            pdf_data = base64.b64encode(pdf_file.read()).decode("utf-8")
        
        content_blocks = [
            {"type": "text", "text": prompt},
            {
                "type": "file",
                "base64": pdf_data,
                "mime_type": "application/pdf",
                # 일부 프로바이더는 파일명 필요
                "extras": {"filename": "리비안_KR_with_table.pdf"}
            }
        ]
    
    message = HumanMessage(content_blocks=content_blocks)
    return model.invoke([message])

# 실행 예시
result = process_pdf_document(
    "data/리비안_KR_with_table.pdf",
    "이 문서의 주요 내용을 요약해주세요"
)

print(result.content)

### 5.2 오디오 처리

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
import base64

# 오디오 지원 모델
gemini_model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    temperature=0,
)

def process_audio(audio_source: str, prompt: str = "이 오디오를 텍스트로 변환해주세요"):
    """
    최신 langchain 메시지 content_blocks 규격 적용.
    :param audio_source: 로컬 오디오 파일 경로 또는 http url
    :param prompt: 프롬프트 메시지
    :return: 텍스트 변환 결과
    """

    if audio_source.startswith("http"):
        content_blocks = [
            {"type": "text", "text": prompt},
            {"type": "audio", "url": audio_source, "mime_type": "audio/mpeg"}
        ]
    else:
        with open(audio_source, "rb") as audio_file:
            audio_data = base64.b64encode(audio_file.read()).decode("utf-8")
        content_blocks = [
            {"type": "text", "text": prompt},
            {
                "type": "audio",
                "base64": audio_data,
                "mime_type": "audio/mpeg"
            }
        ]

    message = HumanMessage(content_blocks=content_blocks)
    result = gemini_model.invoke([message])
    return result  # AIMessage 타입, .content_blocks 또는 .content 사용 가능

# 실행 예시
result = process_audio(
    "data/meeting_sample.mp3", 
    "이 오디오를 텍스트로 변환해주세요"
    )
print(result.content)

### 5.3 비디오 처리

In [ ]:
# 비디오 지원 모델
gemini_model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    temperature=0,
)

def process_video(video_source: str, prompt: str):
    """비디오 처리 (v1.0 표준 콘텐츠 블록)"""
    
    if video_source.startswith("http"):
        content_blocks = [
            {"type": "text", "text": prompt},
            {"type": "video", "url": video_source, "mime_type": "video/mp4"}
        ]
    else:
        with open(video_source, "rb") as video_file:
            video_data = base64.b64encode(video_file.read()).decode("utf-8")
        
        content_blocks = [
            {"type": "text", "text": prompt},
            {
                "type": "video",
                "base64": video_data,
                "mime_type": "video/mp4"
            }
        ]
    
    message = HumanMessage(content_blocks=content_blocks)
    return gemini_model.invoke([message])

# 실행 예시
result = process_video(
    "data/bikes.mp4", 
    "이 비디오에서 일어나는 주요 이벤트를 시간순으로 설명해주세요"
    )

print(result.content)

---

## 6. 표준 콘텐츠 직렬화

- 표준 콘텐츠 블록은 기본적으로 `content` 속성에 직렬화되지 **않습니다**. 
- 외부 애플리케이션에서 표준 콘텐츠 블록 표현이 필요한 경우 별도의 **직렬화** 처리가 필요합니다.

    - **사용해야 할 때**

        - API 엔드포인트 구현
        - 프론트엔드와 통신
        - 멀티 프로바이더 지원
        - 로깅/분석 시스템
        - JSON 직렬화 필요

    - **불필요한 경우**

        - 내부 처리만 수행
        - 단일 프로바이더만 사용
        - 최대 성능 필요



- **핵심 개념**

    ```python
    # 기본 동작 (output_version 미설정)
    response.content         → 프로바이더 네이티브 형식 (예: 문자열)
    response.content_blocks  → LangChain 표준 형식 (항상)

    # 직렬화 활성화 (output_version="v1")
    response.content         → LangChain 표준 형식 ✅
    response.content_blocks  → LangChain 표준 형식 ✅

    # 결과
    response.content == response.content_blocks  # True
    ```


- **비교표**

    | 항목 | 기본 | 직렬화 활성화 |
    |------|------|---------------|
    | `content` 형식 | 프로바이더별 | 표준 |
    | `content_blocks` 형식 | 표준 | 표준 |
    | 프로바이더 전환 | 코드 수정 필요 | 불필요 |
    | 성능 | ⚡ 빠름 | 약간 느림 |
    | 외부 통합 | 어려움 | 쉬움 |



In [ ]:
# 직렬화 없이 사용하는 경우 (기존 방법)
model = init_chat_model("gpt-4o-mini")
response = model.invoke("AI를 설명해주세요")
print(response.content)  # 프로바이더 네이티브 형식

In [ ]:
print(response.content_blocks)  # 표준 형식

In [ ]:
# 직렬화 적용: response.content에 표준 콘텐츠 블록이 직렬화됨
model = init_chat_model("gpt-4o-mini", output_version="v1")
response = model.invoke("AI를 설명해주세요")
print(response.content)  # 표준 형식으로 직렬화됨

In [ ]:
print(response.content_blocks)  # 표준 형식 

In [ ]:
%%writefile fastapi_example.py
# encoding: utf-8
from fastapi import FastAPI
from pydantic import BaseModel
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

app = FastAPI()

model = init_chat_model("gpt-4o-mini", output_version="v1")

@app.post("/query")
async def query(text: str):
    response = model.invoke(text)
    
    return {
        "content": response.content,  # ✅ 표준 형식 보장
        "model": "gpt-4o-mini",
        "tokens": response.usage_metadata["total_tokens"]
    }

In [ ]:
# 노트북 셀에서 백그라운드로 실행
import uvicorn
import threading
from fastapi_example import app

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# 백그라운드 스레드로 실행
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

import time
time.sleep(2)  # 서버 시작 대기
print("✅ 서버가 http://localhost:8000 에서 실행 중입니다")

In [ ]:
import requests

# POST 요청 테스트
response = requests.post(
    "http://localhost:8000/query",
    params={"text": "인공지능에 대해 간단히 설명해주세요"}
)

print("Status Code:", response.status_code)
print("Response:", response.json())

- 서버 종료

    ```bash
    # 포트 8000을 사용하는 프로세스 찾기
    lsof -ti:8000 | xargs kill -9

    # 또는 (macOS/Linux)
    kill $(lsof -t -i:8000)
    ``

### 7. content_blocks 속성 활용

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    include_thoughts=True,
)

response = llm.invoke("How many 'r's are in the word 'strawberry'?")

for block in response.content_blocks:
    if block["type"] == "text":
        print(f"텍스트: {block['text']}")
    
    elif block["type"] == "reasoning":
        print(f"추론 과정: {block['reasoning']}")


---

### [실습] LangChain과 Google Gemini를 활용한 멀티모달 애플리케이션 만들기

- **실습 시나리오**
    - 사용자가 이미지를 업로드하면, 이미지의 내용을 간략하게 설명(요약)해주는 파이프라인을 LangChain 기반으로 구현합니다.
    - 또한, 설명 결과와 reasoning(추론 과정) 블록이 함께 출력되게 하세요.

- **필수 구현 요소**
    1. LangChain의 GoogleGenerativeAI 모델(ChatGoogleGenerativeAI)을 사용합니다.
    2. 이미지를 파일로 읽어서 입력 메시지에 multimodal 형식으로 첨부합니다.
    3. 모델의 응답에서 `content_blocks` 속성을 이용해 설명(텍스트)과 reasoning(추론)의 내용을 구분해서 출력하세요.
    4. 적절한 이미지를 준비해서 테스트해보세요.

- **참고**: [ChatGoogleGenerativeAI 문서](https://reference.langchain.com/python/integrations/langchain_google_genai/)


In [ ]:
# 여기에 코드를 작성하세요.